# 21: PatchCore with EfficientNet-B0 on the 50k / 5% benchmark

Goal:

- adapt the CT EfficientNet-B0 PatchCore prototype to the repo's main `metadata_50k_5pct.csv` protocol
- keep the benchmark split used in the report:
  - `train`: normal only
  - `val`: normal only
  - `test`: `5000` normal + `250` anomaly
- report AUROC / AUPRC and a deployment-style threshold chosen from validation-normal scores only

Notes:

- this is intentionally an exploratory notebook
- it reuses the EfficientNet-B0 + fused mid/deep feature idea from the CT prototype
- unlike the CT notebook, it does **not** tune the operating threshold using labeled anomalies
- unlike the CT notebook, it follows the repo's report-style raw-score thresholding instead of train-score z-normalization

In [ ]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation import summarize_threshold_metrics, sweep_threshold_metrics

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "training" / "train_patchcore_resnet50.toml"
base_config = load_toml(BASE_CONFIG_PATH)

SEED = int(base_config["run"].get("seed", 42))
IMAGE_SIZE = int(base_config["data"].get("image_size", 64))
MODEL_INPUT_SIZE = 224
METADATA_PATH = REPO_ROOT / base_config["data"]["metadata_csv"]

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
BATCH_SIZE = 128 if torch.cuda.is_available() else 16

MEMORY_BANK_MAX_PATCHES = 240_000 if torch.cuda.is_available() else 80_000
SCORE_CHUNK = 1024 if torch.cuda.is_available() else 512
PATCHCORE_NN_K = 3
TOPK_PATCH_RATIO = 0.02
EFFNET_MID_FEATURE_IDX = 3
EFFNET_DEEP_FEATURE_IDX = 6
PATCH_EMBED_DIM = 512
USE_AMP = torch.cuda.is_available()

THRESHOLD_QUANTILE = 0.95

# Optional smoke-run limits. Leave as None for the full benchmark split.
TRAIN_IMAGE_LIMIT = None
VAL_IMAGE_LIMIT = None
TEST_IMAGE_LIMIT = None

OUTPUT_DIR = REPO_ROOT / "artifacts" / "x64" / "patchcore_efficientnet_b0_5pct"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "metadata_path": str(METADATA_PATH.relative_to(REPO_ROOT)),
        "image_size": IMAGE_SIZE,
        "model_input_size": MODEL_INPUT_SIZE,
        "batch_size": BATCH_SIZE,
        "memory_bank_max_patches": MEMORY_BANK_MAX_PATCHES,
        "topk_patch_ratio": TOPK_PATCH_RATIO,
        "patchcore_nn_k": PATCHCORE_NN_K,
        "threshold_quantile": THRESHOLD_QUANTILE,
        "threshold_policy": "val_normal_quantile_raw",
        "output_dir": str(OUTPUT_DIR.relative_to(REPO_ROOT)),
    }
)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)

set_seed(SEED)
device = resolve_device(base_config["training"].get("device", "auto"))
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

device

In [ ]:
metadata = pd.read_csv(METADATA_PATH)
display(metadata.head())
display(metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index())

In [ ]:
class EfficientNetWaferDataset(Dataset):
    def __init__(self, base_dataset: WaferMapDataset) -> None:
        self.base_dataset = base_dataset
        self.metadata = base_dataset.metadata.reset_index(drop=True).copy()

    def __len__(self) -> int:
        return len(self.base_dataset)

    @staticmethod
    def preprocess_wafer_tensor(x: torch.Tensor, size: int) -> torch.Tensor:
        # Processed arrays store wafer states as {0.0, 0.5, 1.0}; map them back to {0, 1, 2}.
        states = torch.clamp(torch.round(x * 2.0), 0, 2).to(dtype=torch.long).squeeze(0)
        one_hot = F.one_hot(states, num_classes=3).permute(2, 0, 1).to(dtype=torch.float32)
        if one_hot.shape[-1] != size or one_hot.shape[-2] != size:
            one_hot = F.interpolate(
                one_hot.unsqueeze(0),
                size=(size, size),
                mode="nearest",
            ).squeeze(0)
        return one_hot

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        x, y = self.base_dataset[index]
        x = self.preprocess_wafer_tensor(x, MODEL_INPUT_SIZE)
        return x, y

def maybe_limit_dataset(dataset: Dataset, limit: int | None) -> Dataset:
    if limit is None or len(dataset) <= limit:
        return dataset
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(len(dataset), generator=generator)[:limit].tolist()
    return Subset(dataset, indices)

def metadata_for_dataset(dataset: Dataset) -> pd.DataFrame:
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        if not hasattr(base_dataset, "metadata"):
            raise AttributeError("Subset base dataset does not expose metadata.")
        return base_dataset.metadata.iloc[list(dataset.indices)].reset_index(drop=True).copy()
    if not hasattr(dataset, "metadata"):
        raise AttributeError("Dataset does not expose metadata.")
    return dataset.metadata.reset_index(drop=True).copy()

train_base_dataset = WaferMapDataset(METADATA_PATH, split="train", image_size=IMAGE_SIZE)
val_base_dataset = WaferMapDataset(METADATA_PATH, split="val", image_size=IMAGE_SIZE)
test_base_dataset = WaferMapDataset(METADATA_PATH, split="test", image_size=IMAGE_SIZE)

train_dataset = EfficientNetWaferDataset(train_base_dataset)
val_dataset = EfficientNetWaferDataset(val_base_dataset)
test_dataset = EfficientNetWaferDataset(test_base_dataset)

train_dataset = maybe_limit_dataset(train_dataset, TRAIN_IMAGE_LIMIT)
val_dataset = maybe_limit_dataset(val_dataset, VAL_IMAGE_LIMIT)
test_dataset = maybe_limit_dataset(test_dataset, TEST_IMAGE_LIMIT)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f"train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
class EfficientNetPatchCoreModel(nn.Module):
    def __init__(
        self,
        *,
        model_input_size: int = 224,
        mid_feature_idx: int = 3,
        deep_feature_idx: int = 6,
        patch_embed_dim: int = 512,
        topk_ratio: float = 0.02,
        nn_k: int = 3,
        query_chunk_size: int = 1024,
    ) -> None:
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features
        self.mid_feature_idx = int(mid_feature_idx)
        self.deep_feature_idx = int(deep_feature_idx)
        self.patch_embed_dim = int(patch_embed_dim)
        self.topk_ratio = float(topk_ratio)
        self.nn_k = int(nn_k)
        self.query_chunk_size = int(query_chunk_size)

        with torch.inference_mode():
            dummy = torch.zeros(1, 3, model_input_size, model_input_size)
            x = dummy
            f_mid = None
            f_deep = None
            for i, block in enumerate(self.features):
                x = block(x)
                if i == self.mid_feature_idx:
                    f_mid = x
                if i == self.deep_feature_idx:
                    f_deep = x

        if f_mid is None or f_deep is None:
            raise ValueError(
                f"Invalid EfficientNet feature indices: mid={self.mid_feature_idx}, deep={self.deep_feature_idx}"
            )

        in_dim = int(f_mid.shape[1] + f_deep.shape[1])
        self.patch_grid = tuple(int(dim) for dim in f_mid.shape[-2:])
        self.feature_dim = self.patch_embed_dim
        self.proj = nn.Linear(in_dim, self.patch_embed_dim, bias=False)
        self.register_buffer("memory_bank", torch.empty(0, self.feature_dim))

        for parameter in self.features.parameters():
            parameter.requires_grad = False
        for parameter in self.proj.parameters():
            parameter.requires_grad = False

    @property
    def patches_per_image(self) -> int:
        return int(self.patch_grid[0] * self.patch_grid[1])

    def forward_feature_maps(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        f_mid = None
        f_deep = None
        for i, block in enumerate(self.features):
            x = block(x)
            if i == self.mid_feature_idx:
                f_mid = x
            if i == self.deep_feature_idx:
                f_deep = x
        if f_mid is None or f_deep is None:
            raise RuntimeError("Failed to collect EfficientNet feature maps.")
        return f_mid, f_deep

    def patch_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                f_mid, f_deep = self.forward_feature_maps(x)
                f_deep_up = F.interpolate(
                    f_deep,
                    size=f_mid.shape[-2:],
                    mode="bilinear",
                    align_corners=False,
                )
                emb = torch.cat([f_mid, f_deep_up], dim=1)
                emb = emb.permute(0, 2, 3, 1).reshape(-1, emb.shape[1])
                emb = self.proj(emb)
            emb = F.normalize(emb.float(), p=2, dim=1)
        return emb.reshape(x.shape[0], self.patches_per_image, self.feature_dim)

    def set_memory_bank(self, memory_bank: torch.Tensor) -> None:
        if memory_bank.ndim != 2 or memory_bank.shape[1] != self.feature_dim:
            raise ValueError(
                f"memory_bank must have shape (N, {self.feature_dim}), got {tuple(memory_bank.shape)}"
            )
        normalized = F.normalize(memory_bank.to(dtype=torch.float32), p=2, dim=1)
        self.memory_bank = normalized.to(device=self.memory_bank.device, dtype=self.memory_bank.dtype)

    def nearest_patch_distances(self, patch_embeddings: torch.Tensor) -> torch.Tensor:
        if self.memory_bank.numel() == 0:
            raise ValueError("memory_bank is empty.")

        batch_size, patch_count, _ = patch_embeddings.shape
        flat_queries = patch_embeddings.reshape(-1, self.feature_dim)
        bank_t = self.memory_bank.t().contiguous()
        outputs = []

        for start in range(0, flat_queries.shape[0], self.query_chunk_size):
            query_chunk = flat_queries[start : start + self.query_chunk_size]
            sim = query_chunk @ bank_t
            k = min(self.nn_k, sim.shape[1])
            best_sim = sim.topk(k=k, dim=1).values
            dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best_sim, min=0.0))
            outputs.append(dist.mean(dim=1))

        return torch.cat(outputs, dim=0).reshape(batch_size, patch_count)

    def reduce_patch_distances(self, patch_distances: torch.Tensor) -> torch.Tensor:
        topk = max(1, int(round(patch_distances.shape[1] * self.topk_ratio)))
        topk = min(topk, patch_distances.shape[1])
        return torch.topk(patch_distances, k=topk, dim=1).values.mean(dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patch_embeddings = self.patch_embeddings(x)
        patch_distances = self.nearest_patch_distances(patch_embeddings)
        return self.reduce_patch_distances(patch_distances)

model = EfficientNetPatchCoreModel(
    model_input_size=MODEL_INPUT_SIZE,
    mid_feature_idx=EFFNET_MID_FEATURE_IDX,
    deep_feature_idx=EFFNET_DEEP_FEATURE_IDX,
    patch_embed_dim=PATCH_EMBED_DIM,
    topk_ratio=TOPK_PATCH_RATIO,
    nn_k=PATCHCORE_NN_K,
    query_chunk_size=SCORE_CHUNK,
).to(device).eval()

print(
    {
        "patches_per_image": model.patches_per_image,
        "feature_dim": model.feature_dim,
        "patch_grid": model.patch_grid,
    }
)

In [ ]:
def collect_memory_bank(
    model: EfficientNetPatchCoreModel,
    dataloader: DataLoader,
    device: torch.device,
    target_size: int,
    seed: int,
) -> torch.Tensor:
    if target_size <= 0:
        raise ValueError("target_size must be positive")

    patch_batches: list[torch.Tensor] = []
    total_seen_patches = 0
    estimated_total_patches = model.patches_per_image * len(dataloader.dataset)
    sample_ratio = min(1.0, target_size / estimated_total_patches)
    generator = torch.Generator().manual_seed(seed)

    print(
        {
            "estimated_total_patches": int(estimated_total_patches),
            "sample_ratio": round(sample_ratio, 6),
        }
    )

    model.eval()
    progress = tqdm(dataloader, desc="Building memory bank", total=len(dataloader))
    with torch.inference_mode():
        for inputs, labels in progress:
            inputs = inputs.to(device, non_blocking=PIN_MEMORY)
            labels = labels.to(device)
            normal_mask = labels == 0
            if not torch.any(normal_mask):
                continue

            embeddings = model.patch_embeddings(inputs[normal_mask]).reshape(-1, model.feature_dim)
            total_seen_patches += int(embeddings.shape[0])
            embeddings = embeddings.cpu()

            if sample_ratio < 1.0:
                keep_n = max(1, int(round(embeddings.shape[0] * sample_ratio)))
                keep_idx = torch.randperm(embeddings.shape[0], generator=generator)[:keep_n]
                embeddings = embeddings[keep_idx]

            patch_batches.append(embeddings)
            progress.set_postfix(seen_patches=total_seen_patches, kept_batches=len(patch_batches))

    if not patch_batches:
        raise ValueError("Could not build memory bank because no normal embeddings were collected.")

    memory_bank = torch.cat(patch_batches, dim=0)
    print({"observed_raw_patches": int(total_seen_patches), "sampled_before_trim": int(memory_bank.shape[0])})

    if memory_bank.shape[0] > target_size:
        keep_gen = torch.Generator().manual_seed(seed)
        keep = torch.randperm(memory_bank.shape[0], generator=keep_gen)[:target_size]
        memory_bank = memory_bank[keep]

    return memory_bank

memory_bank = collect_memory_bank(
    model=model,
    dataloader=train_loader,
    device=device,
    target_size=MEMORY_BANK_MAX_PATCHES,
    seed=SEED,
)
model.set_memory_bank(memory_bank.to(device))
print({"final_memory_bank_size": int(model.memory_bank.shape[0]), "device": str(model.memory_bank.device)})

In [ ]:
def collect_scores(
    model: EfficientNetPatchCoreModel,
    dataloader: DataLoader,
    device: torch.device,
    desc: str,
) -> pd.DataFrame:
    rows = []
    model.eval()
    progress = tqdm(dataloader, desc=desc, total=len(dataloader))
    with torch.inference_mode():
        for inputs, labels in progress:
            inputs = inputs.to(device, non_blocking=PIN_MEMORY)
            scores = model(inputs).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                rows.append({"score": float(score), "is_anomaly": int(label)})
            progress.set_postfix(rows=len(rows))
    return pd.DataFrame(rows)

val_scores_df = collect_scores(model, val_loader, device, desc="Scoring val")
test_scores_df = collect_scores(model, test_loader, device, desc="Scoring test")

val_normal_scores = val_scores_df.loc[val_scores_df["is_anomaly"] == 0, "score"]
threshold = float(val_normal_scores.quantile(THRESHOLD_QUANTILE))

labels = test_scores_df["is_anomaly"].to_numpy()
scores = test_scores_df["score"].to_numpy()
metrics = summarize_threshold_metrics(labels, scores, threshold)
threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)

summary = {
    "name": "PatchCore-EfficientNetB0-5pct",
    "metadata_csv": str(METADATA_PATH.relative_to(REPO_ROOT).as_posix()),
    "memory_bank_size": int(model.memory_bank.shape[0]),
    "patches_per_image": int(model.patches_per_image),
    "feature_dim": int(model.feature_dim),
    "threshold_quantile": float(THRESHOLD_QUANTILE),
    "threshold_policy": "val_normal_quantile_raw",
    "threshold": float(threshold),
    "precision": float(metrics["precision"]),
    "recall": float(metrics["recall"]),
    "f1": float(metrics["f1"]),
    "auroc": float(metrics["auroc"]),
    "auprc": float(metrics["auprc"]),
    "best_sweep_threshold": float(best_sweep["threshold"]),
    "best_sweep_precision": float(best_sweep["precision"]),
    "best_sweep_recall": float(best_sweep["recall"]),
    "best_sweep_f1": float(best_sweep["f1"]),
    "predicted_anomalies": int(metrics["predicted_anomalies"]),
    "config": {
        "image_size": IMAGE_SIZE,
        "model_input_size": MODEL_INPUT_SIZE,
        "effnet_mid_feature_idx": EFFNET_MID_FEATURE_IDX,
        "effnet_deep_feature_idx": EFFNET_DEEP_FEATURE_IDX,
        "patch_embed_dim": PATCH_EMBED_DIM,
        "patchcore_nn_k": PATCHCORE_NN_K,
        "patchcore_topk_ratio": TOPK_PATCH_RATIO,
        "memory_bank_max_patches": MEMORY_BANK_MAX_PATCHES,
        "score_chunk": SCORE_CHUNK,
    },
}

display(pd.DataFrame([summary]))

In [ ]:
eval_dir = OUTPUT_DIR / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

val_scores_df.to_csv(eval_dir / "val_scores.csv", index=False)
test_scores_df.to_csv(eval_dir / "test_scores.csv", index=False)
threshold_sweep_df.to_csv(eval_dir / "threshold_sweep.csv", index=False)
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

checkpoint = {
    "model_state_dict": model.state_dict(),
    "memory_bank": model.memory_bank.detach().cpu(),
    "summary": summary,
}
torch.save(checkpoint, OUTPUT_DIR / "best_model.pt")

print(f"Saved outputs to: {OUTPUT_DIR.relative_to(REPO_ROOT)}")

In [ ]:
print(
    {
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "auroc": metrics["auroc"],
        "auprc": metrics["auprc"],
    }
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cm = np.array(metrics["confusion_matrix"], dtype=int)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["pred normal", "pred anomaly"],
    yticklabels=["true normal", "true anomaly"],
    ax=axes[0],
)
axes[0].set_title("EfficientNet-B0 PatchCore Confusion Matrix")

sns.kdeplot(
    test_scores_df.loc[test_scores_df["is_anomaly"] == 0, "score"],
    label="Normal test",
    fill=True,
    alpha=0.35,
    ax=axes[1],
)
sns.kdeplot(
    test_scores_df.loc[test_scores_df["is_anomaly"] == 1, "score"],
    label="Defect test",
    fill=True,
    alpha=0.35,
    ax=axes[1],
)
axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold {threshold:.3f}")
axes[1].set_title("EfficientNet-B0 PatchCore Score Distribution")
axes[1].set_xlabel("Anomaly score (raw)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
test_analysis_df = metadata_for_dataset(test_dataset)
if len(test_analysis_df) != len(test_scores_df):
    raise ValueError("Metadata and test-score lengths do not match.")

test_analysis_df["score"] = test_scores_df["score"].to_numpy()
test_analysis_df["predicted_anomaly"] = (test_analysis_df["score"] > threshold).astype(int)

defect_breakdown = (
    test_analysis_df[test_analysis_df["is_anomaly"] == 1]
    .groupby("defect_type")
    .agg(
        count=("is_anomaly", "size"),
        recall=("predicted_anomaly", "mean"),
        mean_score=("score", "mean"),
    )
    .sort_values(["recall", "count"], ascending=[False, False])
)

display(defect_breakdown)
display(test_analysis_df.head())